# **03_Data_Cleaning**

This notebook cleans and prepares Google Play Store reviews from four Indonesian ride-hailing platforms (Grab, Gojek, Maxim, InDrive). Starting from 100,000 raw rows, it removes duplicates, standardizes column names and data types, and drops rows with missing or invalid content. It also enriches the data with derived features such as normalized text, temporal breakdowns, review length metrics, and a star-rating-based sentiment label. The final output is **66,878 clean rows** saved to `combined_reviews_clean.csv`, ready for EDA and modelling.

## **1. Import Library**

Importing all necessary libraries and defining key constants. `pandas` and `numpy` handle data manipulation, while `re` is used for regex-based text processing. 

In [ ]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

Setup OK


## **2. Environment Setup & Configuration**

Two `Path` objects are defined: `DATA_DIR` pointing to the raw CSV files, and `CLEAN_DIR` for the cleaned output. The list `PLATFORMS` declares the four ride hailing apps being analyzed. The `CLEAN_DIR` is created automatically if it doesn't already exist.

In [ ]:
DATA_DIR = Path('../data/raw/google_play_reviews')
CLEAN_DIR = Path('../data/clean')
CLEAN_DIR.mkdir(parents=True, exist_ok=True)

PLATFORMS = ['grab', 'gojek', 'maxim', 'indrive']
print("Setup OK")

## **3. Load Data**

This cell iterates over each platform name, constructs the expected file path, and loads the CSV if it exists. A `platform` column is added to each individual DataFrame to preserve the source label. All four platform DataFrames are then concatenated into a single unified DataFrame `df_raw`. The total row count is printed as a sanity check.

In [ ]:
dfs = {}
for p in PLATFORMS:
    fp = DATA_DIR / f'{p}_reviews.csv'
    if fp.exists():
        dfs[p] = pd.read_csv(fp, parse_dates=['at', 'repliedAt'])
        dfs[p]['platform'] = p

df_raw = pd.concat(dfs.values(), ignore_index=True)
print(f"Raw: {len(df_raw):,} rows")

Raw: 100,000 rows


## **4. Duplicate Removal (Two Passes)**

This cell removes duplicates in two stages. The first pass drops rows with a duplicate `reviewId`, which are exact structural duplicates. The second pass removes rows where the same review `content` text appears more than once for the same `platform`, catching cases where different review IDs may still carry identical review text. Both passes use `keep='first'` to retain the earliest occurrence. Row counts before and after each pass are printed.

In [4]:
before = len(df_raw)
df = df_raw.drop_duplicates(subset=['reviewId'], keep='first')
print(f"Duplicates removed: {before - len(df):,} | Remaining: {len(df):,}")

# Also drop fully identical content (same text + same platform)
before2 = len(df)
df = df.drop_duplicates(subset=['content', 'platform'], keep='first')
print(f"Same-text dupes removed: {before2 - len(df):,} | Remaining: {len(df):,}")

Duplicates removed: 0 | Remaining: 100,000
Same-text dupes removed: 32,963 | Remaining: 67,037


## **4. Column Renaming & Data Type Casting**

This cell standardizes column names from camelCase to snake_case for consistency. It then enforces correct data types: `score` is cast to a nullable 8-bit integer (`Int8`) to save memory; `thumbs_up` is coerced to numeric with null values filled as `0`; both date columns (`at`, `replied_at`) are parsed as `datetime64`. The final dtype summary is printed to confirm correctness.

In [5]:
df = df.rename(columns={
    'reviewId': 'review_id',
    'userName': 'user_name',
    'thumbsUpCount': 'thumbs_up',
    'reviewCreatedVersion': 'app_version',
    'replyContent': 'reply_content',
    'repliedAt': 'replied_at',
})

df['score'] = pd.to_numeric(df['score'], errors='coerce').astype('Int8')
df['thumbs_up'] = pd.to_numeric(df['thumbs_up'], errors='coerce').fillna(0).astype(int)
df['at'] = pd.to_datetime(df['at'], errors='coerce')
df['replied_at'] = pd.to_datetime(df['replied_at'], errors='coerce')

print(df.dtypes)

review_id                object
user_name                object
content                  object
score                      Int8
thumbs_up                 int64
app_version              object
at               datetime64[ns]
reply_content            object
replied_at       datetime64[ns]
platform                 object
dtype: object


## **5. Drop Invalid / Empty Rows**

This cell removes rows that are unfit for analysis. First, rows with null `score` or `at` (timestamp) are dropped, these are critical fields without which a review cannot be meaningfully used. Next, rows with null or very short `content` (less than 3 characters after stripping whitespace) are filtered out, as they carry no linguistic signal. A new column `content_clean` is created to hold the stripped version of the raw content.

In [6]:
before = len(df)

# Drop rows with null score or date
df = df.dropna(subset=['score', 'at'])

# Drop reviews with null or too-short content
df = df[df['content'].notna()]
df['content_clean'] = df['content'].str.strip()
df = df[df['content_clean'].str.len() >= 3]

print(f"Dropped {before - len(df):,} invalid rows | Remaining: {len(df):,}")

Dropped 159 invalid rows | Remaining: 66,878


## **6. Text Normalization**

This cell defines and applies a lightweight text normalization function tailored for Indonesian-language reviews. The function lowercases all text, removes URLs, collapses excessive character repetitions (e.g., `!!!!!` → `!!`, `wkwkwkwk` → `wkwk`), and normalizes whitespace. Notably, emojis are intentionally preserved as they are useful sentiment signals. The normalized text is stored in a new column `content_normalized`. A sample of 5 rows is printed to verify the transformation.

In [7]:
def normalize_text(text):
    """
    Light normalization for Indonesian reviews:
    - Lowercase
    - Remove URLs
    - Remove excessive punctuation repetition (!!!!! → !)
    - Normalize whitespace
    - Keep emojis (useful sentiment signal)
    """
    if not isinstance(text, str):
        return ''
    text = text.lower()
    text = re.sub(r'https?://\S+|www\.\S+', '', text)         # remove URLs
    text = re.sub(r'(.)\1{3,}', r'\1\1', text)                # e.g. wkwkwk → wkwk, !!!! → !!
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['content_normalized'] = df['content_clean'].apply(normalize_text)
print("Sample normalized:")
print(df[['content_clean', 'content_normalized']].head(5).to_string())

Sample normalized:
                                                                                                                                                                                            content_clean                                                                                                                                                                                      content_normalized
0                                                                                                                                                                                                   bagus                                                                                                                                                                                                   bagus
1  perbaikan tuh aplikasi lu udah ongkir mahal voucher lu gajelas tiba-tiba ga bisa digunain padahal masa habisnya masih lama mana ngelag lagi lu makin lama gini ya makin amit-a

## **7. Feature Engineering**

This cell engineers several new columns to support downstream EDA and modelling. Temporal features (`year_month`, `year`, `month`, `day_of_week`) are extracted from the `at` timestamp. Text length features (review_len, word_count) quantify review verbosity. A binary `has_reply` flag indicates whether the app developer responded. Finally, `sentiment_label_score` maps the numeric star rating into a coarse three-class label (`positive` for 4–5 stars, `neutral` for 3, `negative` for 1–2). A sample of 10 rows is printed.

In [8]:
# Derived columns useful for EDA & modelling
df['review_len'] = df['content_clean'].str.len()
df['word_count'] = df['content_clean'].str.split().str.len()
df['has_reply'] = df['reply_content'].notna().astype(int)
df['year_month'] = df['at'].dt.to_period('M').astype(str)
df['year'] = df['at'].dt.year
df['month'] = df['at'].dt.month
df['day_of_week'] = df['at'].dt.day_name()

# Coarse sentiment label from score (used as quick baseline)
def score_to_label(s):
    if s >= 4: return 'positive'
    if s == 3: return 'neutral'
    return 'negative'

df['sentiment_label_score'] = df['score'].apply(score_to_label)

print(df[['platform','score','sentiment_label_score','review_len','has_reply']].head(10))


  platform  score sentiment_label_score  review_len  has_reply
0     grab      5              positive           5          0
1     grab      2              negative         198          1
2     grab      1              negative         110          1
3     grab      5              positive           5          0
4     grab      5              positive           9          0
5     grab      1              negative         178          1
6     grab      1              negative         152          1
7     grab      1              negative          81          1
8     grab      5              positive         102          0
9     grab      1              negative           5          1


## **8. Per-Platform Quality Summary & Null Audit**

This cell produces a final quality report before saving. It groups the cleaned DataFrame by **platform** and aggregates four key metrics: total review count, average star score, and the percentage of positive and negative reviews. This gives a quick at-a-glance comparison of how each platform performs in user sentiment. A null audit is also printed, showing only columns that still contain missing values, which are expected (e.g., **reply_content** is null when a developer hasn't replied).

In [ ]:
print("=== Per-platform counts ===")
print(df.groupby('platform').agg(
    reviews=('review_id','count'),
    avg_score=('score', lambda x: round(x.mean(), 2)),
    pct_positive=('sentiment_label_score', lambda x: round((x=='positive').mean()*100, 1)),
    pct_negative=('sentiment_label_score', lambda x: round((x=='negative').mean()*100, 1)),
).to_string())

print(f"\nFinal nulls:\n{df.isnull().sum()[df.isnull().sum()>0]}")

=== Per-platform counts ===
          reviews  avg_score  pct_positive  pct_negative
platform                                                
gojek       18301       3.11          49.6          45.0
grab        16218       3.33          55.4          39.8
indrive     17979       3.37          56.5          37.6
maxim       14380       4.27          80.5          16.2

Final nulls:
user_name            1
app_version      13193
reply_content    41240
replied_at       41240
dtype: int64


## **9. Export Cleaned Dataset to CSV**

This final cell saves the fully cleaned and feature-enriched DataFrame to a CSV file at the path defined by `CLEAN_DIR`. The output file `combined_reviews_clean.csv` contains all 66,878 rows that survived the cleaning pipeline. A confirmation message is printed along with the complete list of column names in the exported file, serving as a final manifest of the dataset's schema.

In [10]:
out_path = CLEAN_DIR / 'combined_reviews_clean.csv'
df.to_csv(out_path, index=False)
print(f"✅ Saved {len(df):,} rows → {out_path}")
print("Columns:", df.columns.tolist())

✅ Saved 66,878 rows → ..\data\clean\combined_reviews_clean.csv
Columns: ['review_id', 'user_name', 'content', 'score', 'thumbs_up', 'app_version', 'at', 'reply_content', 'replied_at', 'platform', 'content_clean', 'content_normalized', 'review_len', 'word_count', 'has_reply', 'year_month', 'year', 'month', 'day_of_week', 'sentiment_label_score']
